# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimanshahid800/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/aimanshahid800/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 181, done.
remote: Counting objects: 100% (181/181), done.
remote: Compressing objects: 100% (137/137), done.
remote: Total 181 (delta 74), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (181/181), 1.98 MiB | 953.00 KiB/s, done.
Resolving deltas: 100% (74/74), done.
/content/flyrank-ml-internship




*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two Paper Findings + My Methodology Questions

**Finding #2 — The Content Performance Curve (decay cliff at 271-365 days)**

- **Label source:** Health score (composite: impressions 30pts + position 30pts + CTR 20pts + scroll depth 20pts), bucketed by content age. This is a direct aggregate comparison from raw GSC/GA4 data — not a model prediction.
- **Does the validation design carry the claim?** Partially. This is an observational, correlational comparison — no train/test split, no holdout. The paper itself flags this as "not evidence that age naturally reverses performance decline on its own." The large sample size (341K pieces) makes the pattern directionally robust, but it can't support a causal claim ("age causes decline"), only a descriptive one ("age and low health co-occur"). My own capstone label (`is_declining`) is similarly just a decline flag, not a causal test — same caution applies to my work.

**Finding — ML Appendix: Feature Importance (avg_position = 43% importance predicting Health Score)**

- **Label source:** Health score again — but critically, Health Score is partly *constructed from* average position (30 of 100 points). So avg_position is both an input to the label and a top predictor of it.
- **Does the validation design carry the claim?** No, not fully. The paper is explicit about this: "importance is descriptive rather than causal" because the target already includes the feature. This is a **circularity risk** — my own w05_model.ipynb has almost the identical pattern (avg_position is the #1 feature importance at 0.375, and `avg_position` also directly informs the `is_declining` label logic via trend direction). This is a fair thing to flag honestly in my own capstone's limitations, not something to hide.

**Takeaway for my own model:** Both findings show that a "confirmed" or high-AUC result can still be observational or partially circular. My capstone's AUC (0.760) is a genuine predictive signal, but I should be careful not to claim it "explains why" pages decline — only that it ranks them usefully.

## 2. My Model Under an Honest Split (Before/After)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My w05_model.ipynb used `GroupShuffleSplit` grouped by `content_id`. Re-checking that choice for this audit: `content_id` is **unique per row** (30,000 unique IDs across 30,000 rows — verified below). Grouping by a column that never repeats gives the exact same partition a plain random split would give. So my Week-5 "grouped split" was not actually testing generalization to anything unseen — it was a random split wearing a grouped-split label.

The entity that genuinely **repeats** in this data is `client_id` (32 clients, from 7,008 rows for the biggest client down to a handful for the smallest). That is the honest grouping key: it asks "does the model work on a client it never trained on?" — which is the real deployment question, since FlyRank would run this on new clients, not new pages from clients already in the training set.

**Before (my original Week-5 choice):** random split and content_id-"grouped" split (shown to be equivalent).
**After (the honest fix):** grouped by `client_id`.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

model_df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
model_df["is_declining"] = (model_df["trend_direction"] == "down").astype(int)
features = ["content_age_days", "impressions_last_30d", "avg_position", "ctr"]
model_df = model_df.dropna(subset=features + ["is_declining"])

print(f"Rows: {len(model_df)}")
print(f"Unique content_id: {model_df['content_id'].nunique()}  <- equals row count, confirms no grouping effect")
print(f"Unique client_id:  {model_df['client_id'].nunique()}  <- the entity that actually repeats")
print(f"Base rate (is_declining=1): {model_df['is_declining'].mean():.3f}")

def fit_eval(train_df, test_df):
    rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, class_weight="balanced")
    rf.fit(train_df[features], train_df["is_declining"])
    auc = roc_auc_score(test_df["is_declining"], rf.predict_proba(test_df[features])[:, 1])
    return rf, auc

# BEFORE #1: naive random split (no grouping at all)
train_r, test_r = train_test_split(model_df, test_size=0.2, random_state=42, stratify=model_df["is_declining"])
_, auc_random = fit_eval(train_r, test_r)

# BEFORE #2: "grouped" by content_id — my original Week-5 approach
gss_c = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss_c.split(model_df, groups=model_df["content_id"]))
train_c, test_c = model_df.iloc[tr_idx], model_df.iloc[te_idx]
_, auc_content_grouped = fit_eval(train_c, test_c)

# AFTER: grouped by client_id — the honest split
gss_cl = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx2, te_idx2 = next(gss_cl.split(model_df, groups=model_df["client_id"]))
train_cl, test_cl = model_df.iloc[tr_idx2], model_df.iloc[te_idx2]
rf_honest, auc_client_grouped = fit_eval(train_cl, test_cl)

print(f"\nTrain clients: {train_cl['client_id'].nunique()}, Test clients (unseen): {test_cl['client_id'].nunique()}")
print(f"\nRandom split AUC:                {auc_random:.3f}")
print(f"content_id-'grouped' split AUC:  {auc_content_grouped:.3f}  (matches random — confirms it was fake grouping)")
print(f"client_id-grouped split AUC:     {auc_client_grouped:.3f}  (the honest number)")

Rows: 30000
Unique content_id: 30000  <- equals row count, confirms no grouping effect
Unique client_id:  32  <- the entity that actually repeats
Base rate (is_declining=1): 0.542

Train clients: 25, Test clients (unseen): 7

Random split AUC:                0.747
content_id-'grouped' split AUC:  0.760  (matches random — confirms it was fake grouping)
client_id-grouped split AUC:     0.627  (the honest number)


**Result:** Random split AUC = 0.747. content_id-"grouped" split AUC = 0.760 — essentially identical to random, confirming that grouping by a column with zero repeats does nothing. The honest fix, grouping by `client_id`, gives AUC = **0.627** — a drop of about **0.12–0.13** from what Week-5 reported.

That gap is itself the finding: roughly a third of the model's apparent skill (0.760 → base rate 0.542 is a 0.218 lift; only about 0.085 of that survives an unseen-client test) came from the model partly memorizing per-client patterns — writing style, industry, average impression volume per client — rather than learning a page-level decline signal that transfers to a brand FlyRank has never seen. 0.627 is still meaningfully better than the base rate and the 0.494 rule-based baseline, so the model isn't worthless — but 0.760 was an inflated number, and my Week-5 notebook reported it as the headline AUC without noticing this.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set — and real failure examples, on the honest split.*

Same hunt as Week 3, applied to my final feature set: `content_age_days`, `impressions_last_30d`, `avg_position`, `ctr`.

**Label:** `is_declining` = 1 if `trend_direction == "down"`, else 0. Per the data dictionary, `trend_direction` (and the `trend_pct` it's built from) is computed as `(impressions_last_30d − impressions_prev_30d) / impressions_prev_30d`.

**Checked each feature for leakage risk:**

| Feature | Leakage risk? | Reasoning |
|---|---|---|
| `content_age_days` | None | Independent of the trend formula — pure metadata (days since creation) |
| `impressions_last_30d` | **Confirmed label-derived feature** | This is literally one of the two terms in the `trend_pct` formula that defines the label. It isn't a future-window leak, but it is a Leakage-Type-1 case (label-derived feature) per the skill's taxonomy, not a "definitional caution" |
| `avg_position` | None | Search ranking is a separate signal from impression trend — not used in `trend_direction`'s formula |
| `ctr` | None | Click-through rate is independent of the impression-trend label |

**The train-with/train-without test (run above, on the honest client-grouped split):** removing `impressions_last_30d` drops AUC from 0.627 to 0.554 — barely above the 0.542 base rate. So a large share of the model's honest, unseen-client skill is coming from a feature that is mathematically one half of how the label itself is built. This is milder than the skill's "collapse from ~1.0 to ~0.7" pattern (there's no single feature towering at near-perfect AUC), but it is the same failure mode in a smaller dose: the model is partly reading a rearranged version of its own answer key, not purely learning independent decline signal.

**Real failure examples (unseen-client test set):** shown above — one false positive (model flagged a page as declining that was actually stable) and one false negative (model missed a real decline), each with their raw feature values, so the failure is inspectable rather than just a count.

**Revised verdict:** this is a real, if partial, leakage issue — not something to soften into "caution, not leakage." The honest claim about this model is narrower than Week-5's: it ranks decline risk using a mix of one genuinely independent signal (`avg_position`, `content_age_days`, `ctr`) and one feature that is definitionally close to the label. Both the leakage and the client-generalization gap found in Section 2 point the same direction: the Week-5 AUC of 0.760 overstated what this model can do on a client FlyRank hasn't seen yet.

**No client names, domains, URLs, or private queries** appear anywhere in the feature set or training data — confirmed clean, same as capstone.ipynb's Section 2. (`client_id` values used above are already the anonymized pseudonym IDs shipped in the CSV, used only for grouping.)

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3: Leakage audit — WITH vs WITHOUT the suspect feature, under the HONEST (client) split

features_no_imp = ["content_age_days", "avg_position", "ctr"]

rf_without = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, class_weight="balanced")
rf_without.fit(train_cl[features_no_imp], train_cl["is_declining"])
auc_without = roc_auc_score(test_cl["is_declining"], rf_without.predict_proba(test_cl[features_no_imp])[:, 1])

print(f"Honest (client-grouped) AUC WITH impressions_last_30d:    {auc_client_grouped:.3f}")
print(f"Honest (client-grouped) AUC WITHOUT impressions_last_30d: {auc_without:.3f}")
print(f"Drop from removing it: {auc_client_grouped - auc_without:.3f}")

importances = pd.DataFrame({
    "feature": features,
    "importance": rf_honest.feature_importances_
}).sort_values("importance", ascending=False)
print("\nFeature importances (honest client-grouped model):")
print(importances.to_string(index=False))

# --- Real failure examples, on the honest split ---
test_cl = test_cl.copy()
test_cl["model_prob"] = rf_honest.predict_proba(test_cl[features])[:, 1]
test_cl["model_pred"] = (test_cl["model_prob"] > 0.5).astype(int)

false_positives = test_cl[(test_cl["model_pred"] == 1) & (test_cl["is_declining"] == 0)]
false_negatives = test_cl[(test_cl["model_pred"] == 0) & (test_cl["is_declining"] == 1)]

print(f"\nUnseen-client test set: {len(test_cl)} rows, {len(false_positives)} false positives, {len(false_negatives)} false negatives")
print("\nFalse positive example (predicted declining, actually stable):")
print(false_positives[["client_id"] + features].head(1).to_string(index=False))
print("\nFalse negative example (predicted stable, actually declining):")
print(false_negatives[["client_id"] + features].head(1).to_string(index=False))

Honest (client-grouped) AUC WITH impressions_last_30d:    0.627
Honest (client-grouped) AUC WITHOUT impressions_last_30d: 0.554
Drop from removing it: 0.073

Feature importances (honest client-grouped model):
             feature  importance
        avg_position    0.353278
    content_age_days    0.296189
impressions_last_30d    0.243013
                 ctr    0.107520

Unseen-client test set: 6163 rows, 1430 false positives, 1038 false negatives

False positive example (predicted declining, actually stable):
        client_id  content_age_days  impressions_last_30d  avg_position  ctr
client_8527a891e2               238                    85          39.8  0.0

False negative example (predicted stable, actually declining):
        client_id  content_age_days  impressions_last_30d  avg_position  ctr
client_4e07408562               445                  2501          20.3 0.05


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**My boldest sentence (from capstone.ipynb Section 4 — Results):**
> "Random Forest AUC = 0.760 — big genuine improvement" over the baseline rule (AUC 0.494).

**Problem with this phrasing:** two problems, both surfaced by this audit. First, "big genuine improvement" implies a settled, general claim that would hold for any FlyRank client — but Section 2 showed that number came from a split that never tested an unseen client; the honest client-grouped AUC is 0.627, not 0.760. Second, Section 3 showed part of even that reduced number is coming from a feature (`impressions_last_30d`) that overlaps with how the label is built. Stacking both corrections, the genuinely trustworthy claim is smaller than Week-5's headline number in two separate ways.

**Rewritten in safe language (observed / measured / directional / decision-support):**
> On a held-out set of clients the model never trained on, the Random Forest **measured** an AUC of 0.627 — above the 0.542 base rate and above the 0.494 rule-based baseline, but well below the 0.760 figure reported in Week 5, which came from a split that did not actually test generalization to a new client. Part of this signal (dropping to 0.554 without it) comes from a feature that is mathematically related to the label's own definition. This is an **observed**, **directional** improvement over the hand-written rule for ranking decline risk on new clients — useful as **decision-support** for choosing which pages to review first — not a validated, general-purpose decline predictor, and not evidence that the model "explains" why pages decline.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.